In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path
import time

In [2]:
BANK_NAME = "Hana Bank"
BANK_CODE = "HANA"

URL = "https://www.kebhana.com/cms/fxd/p022_foreignremit_02.do"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "X-Requested-With": "XMLHttpRequest",
    "X-Prototype-Version": "1.5.1.1",
    "Origin": "https://www.kebhana.com",
    "Referer": "https://www.kebhana.com/cms/fxd/index.do?contentUrl=/cms/fxd/p022_foreignremit_01.do",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
}

SAVE_DIR = Path("./outputs_hana_fx")
SAVE_DIR.mkdir(exist_ok=True)

In [3]:
def generate_quarter_end_dates(start_year=2004, end_year=2019):
    dates = []
    for y in range(start_year, end_year + 1):
        dates.extend([
            f"{y}0331",
            f"{y}0630",
            f"{y}0930",
            f"{y}1231",
        ])
    return dates

target_dates = generate_quarter_end_dates(2004, 2019)
target_dates[:8], target_dates[-4:]

(['20040331',
  '20040630',
  '20040930',
  '20041231',
  '20050331',
  '20050630',
  '20050930',
  '20051231'],
 ['20190331', '20190630', '20190930', '20191231'])

In [4]:
currencies = ["USD", "JPY", "CNY", "EUR", "GBP"]

In [5]:
def fetch_hana_html(query_date: str, currency: str, session: requests.Session) -> str:
    payload = {
        "ajax": "true",
        "irt": "0100108000101|01001080001013300|외화정기예금 |외화정기예금|0",
        "curCd3": currency,
        "inqStrDt": query_date,
        "inqEndDt": query_date,
        "cd": "0100108000101",
        "irtCd": "01001080001013300",
        "cdNm": "외화정기예금 ",
        "irtCdNm": "외화정기예금",
        "irtTrmStrNo": "",
        "irtTrmEndNo": "",
        "irtTrmEndNo2": "",
        "rangCd": "",
        "hid_key_data": "",
        "hid_enc_data": "",
        "requestTarget": "hanaBodyDiv",
    }

    r = session.post(URL, data=payload, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text

In [6]:
def clean_text(x):
    if x is None:
        return None
    x = str(x).strip()
    return x if x != "" else None

def clean_rate(x):
    x = clean_text(x)
    if x is None:
        return None
    x = x.replace(",", "")
    try:
        return float(x)
    except:
        return None

def parse_hana_table(html: str, target_date: str, currency: str) -> pd.DataFrame:
    soup = BeautifulSoup(html, "lxml")
    table = soup.find("table")

    if table is None:
        return pd.DataFrame(columns=[
            "bank", "bank_code", "target_date", "observed_date",
            "currency", "residency", "maturity_label",
            "term_start", "term_end", "rate", "interest_type",
            "product_group", "unit"
        ])

    tbody = table.find("tbody")
    if tbody is None:
        return pd.DataFrame(columns=[
            "bank", "bank_code", "target_date", "observed_date",
            "currency", "residency", "maturity_label",
            "term_start", "term_end", "rate", "interest_type",
            "product_group", "unit"
        ])

    rows = []
    tr_list = tbody.find_all("tr")

    for tr in tr_list:
        cells = tr.find_all("td")
        vals = [c.get_text(" ", strip=True) for c in cells]

        # 정상 행은 6개 칼럼
        if len(vals) < 5:
            continue

        observed_date = clean_text(vals[0])
        residency_label = clean_text(vals[1])
        term_start = clean_text(vals[2])
        term_end = clean_text(vals[3]) if len(vals) > 3 else None
        rate = clean_rate(vals[4]) if len(vals) > 4 else None
        interest_type = clean_text(vals[5]) if len(vals) > 5 else None

        if residency_label is None:
            continue

        if "거주자" in residency_label and "비거주자" not in residency_label:
            residency = "resident"
        elif "비거주자" in residency_label:
            residency = "nonresident"
        else:
            residency = None

        rows.append({
            "bank": BANK_NAME,
            "bank_code": BANK_CODE,
            "target_date": f"{target_date[:4]}-{target_date[4:6]}-{target_date[6:8]}",
            "observed_date": observed_date.replace(".", "-") if observed_date else None,
            "currency": currency,
            "residency": residency,
            "maturity_label": residency_label,
            "term_start": term_start,
            "term_end": term_end,
            "rate": rate,
            "interest_type": interest_type,
            "product_group": "외화정기예금",
            "unit": "annual % (pretax)",
        })

    return pd.DataFrame(rows)

In [7]:
session = requests.Session()

test_date = "20260325"
test_currency = "USD"

html = fetch_hana_html(test_date, test_currency, session)
print(html[:1500])

df_test = parse_hana_table(html, test_date, test_currency)
print(df_test.shape)
print(df_test.head(20))








<script type="text/javascript">
//<![CDATA[
	$j(document).ready(function() {
	}); 
//]]>
</script>




<h4>외화정기예금, 통화코드 : USD</h4>

	
	
	
<table class="tblBasic leftNone" summary="이율변경일, 거주자구분, 금리기간시작범위, 금리기간 종료범위, 금리, 금리적용이자 지급형테 대한 정보가 있습니다.">
<caption>외화고단위플러스-금리확정형 정보</caption>
<colgroup>
<col width="15%" />
<col width="15%" />
<col width="18%" />
<col width="18%" />
<col width="19%" />
<col width="*" />
</colgroup>
<thead>
<tr>
<th scope="col">이율변경일</th>
<th class="leftLine" scope="col">거주자구분</th>
<th class="leftLine" scope="col">금리기간<br />시작범위</th>
<th class="leftLine" scope="col">금리기간<br />종료범위</th>
<th class="leftLine" scope="col">금리<br/>(연이율,세전 기준)</th>
<th class="leftLine" scope="col">금리적용이자<br />지급형태</th>
</tr>
</thead>
<tbody>
		
			
<tr>
<td class="tc">2026.03.25</td>
<td class="tc">거주자 7일미만</td>
<td class="tc">1일 이상</td>
<td class="tc">7일 미만</td>
<td class="tc">1.9809</td>
<td class="tc"></td>
			
<tr>
<td class="tc">2026.03.25</td>
<td class="tc">거주자 7일이상</td>
<td

In [8]:
session = requests.Session()

test_date = "20260325"
parts = []

for cur in currencies:
    try:
        html = fetch_hana_html(test_date, cur, session)
        df_cur = parse_hana_table(html, test_date, cur)
        parts.append(df_cur)
        print(cur, df_cur.shape)
        time.sleep(0.4)
    except Exception as e:
        print("FAILED:", cur, e)

hana_test_all = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
print(hana_test_all.shape)
hana_test_all.head(30)

USD (12, 13)
JPY (12, 13)
CNY (12, 13)
EUR (12, 13)
GBP (12, 13)
(60, 13)


,bank,bank_code,target_date,observed_date,currency,residency,maturity_label,term_start,term_end,rate,interest_type,product_group,unit
0,Hana Bank,HANA,2026-03-25,2026-03-25,USD,resident,거주자 7일미만,1일 이상,7일 미만,1.9809,None,외화정기예금,annual % (pretax)
1,Hana Bank,HANA,2026-03-25,2026-03-25,USD,resident,거주자 7일이상,7일 이상,1개월 미만,2.8557,None,외화정기예금,annual % (pretax)
2,Hana Bank,HANA,2026-03-25,2026-03-25,USD,resident,거주자 1월이상,1개월 이상,3개월 미만,3.0288,None,외화정기예금,annual % (pretax)
3,Hana Bank,HANA,2026-03-25,2026-03-25,USD,resident,거주자 3월이상,3개월 이상,6개월 미만,3.1172,None,외화정기예금,annual % (pretax)
4,Hana Bank,HANA,2026-03-25,2026-03-25,USD,resident,거주자 6월이상,6개월 이상,12개월 미만,3.0771,None,외화정기예금,annual % (pretax)
5,Hana Bank,HANA,2026-03-25,2026-03-25,USD,resident,거주자 1년제,12개월,NaN,3.0196,None,외화정기예금,annual % (pretax)
6,Hana Bank,HANA,2026-03-25,2026-03-25,USD,nonresident,비거주자 7일미만,1일 이상,7일 미만,2.1087,None,외화정기예금,annual % (pretax)
7,Hana Bank,HANA,2026-03-25,2026-03-25,USD,nonresident,비거주자 7일이상,7일 이상,1개월 미만,3.0399,None,외화정기예금,annual % (pretax)
8,Hana Bank,HANA,2026-03-25,2026-03-25,USD,nonresident,비거주자 1월이상,1개월 이상,3개월 미만,3.0597,None,외화정기예금,annual % (pretax)
9,Hana Bank,HANA,2026-03-25,2026-03-25,USD,nonresident,비거주자 3월이상,3개월 이상,6개월 미만,3.1490,None,외화정기예금,annual % (pretax)


In [9]:
session = requests.Session()

all_parts = []
fail_log = []

for i, d in enumerate(target_dates, start=1):
    for cur in currencies:
        try:
            html = fetch_hana_html(d, cur, session)
            parsed = parse_hana_table(html, d, cur)

            if parsed.empty:
                fail_log.append({
                    "query_date": d,
                    "currency": cur,
                    "reason": "empty_table"
                })
            else:
                all_parts.append(parsed)

            time.sleep(0.4)

        except Exception as e:
            fail_log.append({
                "query_date": d,
                "currency": cur,
                "reason": str(e)
            })

    if i % 5 == 0:
        print(f"{i}/{len(target_dates)} dates done")

hana_all = pd.concat(all_parts, ignore_index=True) if all_parts else pd.DataFrame(columns=[
    "bank", "bank_code", "target_date", "observed_date",
    "currency", "residency", "maturity_label",
    "term_start", "term_end", "rate", "interest_type",
    "product_group", "unit"
])

hana_fail = pd.DataFrame(fail_log)

print("hana_all shape:", hana_all.shape)
print("hana_fail shape:", hana_fail.shape)
hana_all.head()

5/64 dates done
10/64 dates done
15/64 dates done
20/64 dates done
25/64 dates done
30/64 dates done
35/64 dates done
40/64 dates done
45/64 dates done
50/64 dates done
55/64 dates done
60/64 dates done
hana_all shape: (3558, 13)
hana_fail shape: (1, 3)


,bank,bank_code,target_date,observed_date,currency,residency,maturity_label,term_start,term_end,rate,interest_type,product_group,unit
0,Hana Bank,HANA,2004-03-31,2004-03-31,USD,resident,거주자 7일미만,1일 이상,7일 미만,0.5812,None,외화정기예금,annual % (pretax)
1,Hana Bank,HANA,2004-03-31,2004-03-31,USD,resident,거주자 7일이상,7일 이상,1개월 미만,0.5812,None,외화정기예금,annual % (pretax)
2,Hana Bank,HANA,2004-03-31,2004-03-31,USD,resident,거주자 1월이상,1개월 이상,3개월 미만,0.8400,None,외화정기예금,annual % (pretax)
3,Hana Bank,HANA,2004-03-31,2004-03-31,USD,resident,거주자 3월이상,3개월 이상,6개월 미만,1.0100,None,외화정기예금,annual % (pretax)
4,Hana Bank,HANA,2004-03-31,2004-03-31,USD,resident,거주자 6월이상,6개월 이상,NaN,1.1600,None,외화정기예금,annual % (pretax)


In [10]:
if not hana_all.empty:
    hana_all["target_date"] = pd.to_datetime(hana_all["target_date"])
    hana_all["observed_date"] = pd.to_datetime(hana_all["observed_date"], errors="coerce")

    summary = (
        hana_all.groupby(["currency", "residency", "maturity_label"])["rate"]
        .agg(["count", "min", "max"])
        .reset_index()
        .sort_values(["currency", "residency", "maturity_label"])
    )

    print(summary.head(30))
else:
    print("hana_all is empty")

   currency    residency        maturity_label  count     min     max
0       CNY  nonresident  비거주자   위안(중국) 70만 이상     14  0.2000  0.6000
1       CNY  nonresident              비거주자 1년제     35  0.0000  2.2000
2       CNY  nonresident             비거주자 1월이상     63  0.0000  2.2000
3       CNY  nonresident             비거주자 3월이상     63  0.0000  2.2000
4       CNY  nonresident             비거주자 6월이상     63  0.0000  2.2000
5       CNY  nonresident             비거주자 7일미만     63  0.0000  0.6000
6       CNY  nonresident             비거주자 7일이상     64  0.0000  0.2500
7       CNY     resident   거주자   위안(중국) 70만 이상     14  0.2000  0.6000
8       CNY     resident               거주자 1년제     35  0.0000  2.2000
9       CNY     resident              거주자 1월이상     63  0.0000  2.2000
10      CNY     resident              거주자 3월이상     63  0.0000  2.2000
11      CNY     resident              거주자 6월이상     63  0.0000  2.2000
12      CNY     resident              거주자 7일미만     63  0.0000  0.2000
13      CNY     resi

In [11]:
output_path = SAVE_DIR / "hana_fx_all_maturities_2004_2019.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    hana_all.to_excel(writer, sheet_name="raw_all", index=False)
    hana_fail.to_excel(writer, sheet_name="fail_log", index=False)

    if not hana_all.empty:
        summary = (
            hana_all.groupby(["currency", "residency", "maturity_label"])["rate"]
            .agg(["count", "min", "max"])
            .reset_index()
            .sort_values(["currency", "residency", "maturity_label"])
        )
        summary.to_excel(writer, sheet_name="summary", index=False)

print("saved to:", output_path)

saved to: outputs_hana_fx\hana_fx_all_maturities_2004_2019.xlsx
